# Exploration des données et de la couleur (P1)

Ce notebook couvre les tâches 3 à 6 du cahier de P1 :

1. décrire les données ;
2. montrer des patchs par hôpital et par classe ;
3. mesurer les écarts de couleur entre hôpitaux ;
4. essayer `preprocessing.py` sur de vraies images.

Pas besoin de GPU : aucun modèle profond n'est entraîné ici. Les figures et les tableaux sont enregistrés dans `Mon Drive/SPE1_resultats_P1`.

**Règle sur H2.** Le sujet demande de montrer et de comparer les couleurs de chaque hôpital, donc H2 apparaît dans les descriptions. Aucun choix de technique ne doit s'appuyer sur H2, et les comparaisons entre normal et tumoral l'excluent.

## 0. Chargement

In [ ]:
import os, sys, time, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from google.colab import drive
drive.mount("/content/drive")

if not os.path.exists("depot"):
    !git clone --depth 1 https://github.com/kenzo-kouokam/detection-metastases-camelyon17.git depot
sys.path.insert(0, "depot/p1_data")
from charger_donnees import charger_donnees
import preprocessing as pp

df = charger_donnees()
SEED = 42
HOPITAUX = [0, 1, 2, 3, 4]
SORTIE = "/content/drive/MyDrive/SPE1_resultats_P1"
os.makedirs(SORTIE, exist_ok=True)

BLEU, ENCRE, ENCRE_2, GRILLE = "#2a78d6", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRILLE,
                     "axes.labelcolor": ENCRE_2, "text.color": ENCRE, "xtick.color": ENCRE_2, "ytick.color": ENCRE_2,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "figure.dpi": 110})

def image(octets):
    return np.asarray(Image.open(BytesIO(octets)).convert("RGB"))

def sauver(fig, nom):
    fig.savefig(f"{SORTIE}/{nom}.png", dpi=200, bbox_inches="tight")
print(len(df), "patchs")

## 1. Décrire les données (tâche 4)

Combien de patchs dans chaque jeu, et d'où viennent les patchs tumoraux ?

In [ ]:
effectifs = pd.crosstab(df.split_equipe, [df.center, df.label])
effectifs.to_csv(f"{SORTIE}/effectifs_par_jeu.csv")
display(effectifs)

lames = pd.read_csv("depot/p1_data/sorties/audit_lames.csv")
print("Lames par classe et par hôpital")
display(pd.crosstab(lames.center, lames.classe))

concentration = (lames.sort_values("tum_sub", ascending=False).groupby("center")
                 .agg(plus_grosse_lame=("slide", "first"), classe=("classe", "first"), tumoraux_lame=("tum_sub", "first"),
                      tumoraux_hopital=("tum_sub", "sum")))
concentration["part"] = (concentration.tumoraux_lame / concentration.tumoraux_hopital).round(3)
concentration.to_csv(f"{SORTIE}/concentration_par_lame.csv")
print("Part des patchs tumoraux venant de la plus grosse lame")
display(concentration)
print("Part de chaque classe de lame dans les patchs tumoraux")
display((lames.groupby("classe").tum_sub.sum() / lames.tum_sub.sum()).round(4))

## 2. Montrer des patchs (tâche 5)

Une ligne par hôpital : quatre patchs normaux, puis quatre patchs tumoraux.

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(11, 7.2))
for r, h in enumerate(HOPITAUX):
    for lab in (0, 1):
        tirage = df[(df.center == h) & (df.label == lab)].sample(4, random_state=SEED)
        for c, octets in enumerate(tirage.image):
            ax = axes[r, c + 4 * lab]
            ax.imshow(image(octets)); ax.set_xticks([]); ax.set_yticks([])
            for bord in ax.spines.values():
                bord.set_visible(False)
    axes[r, 0].set_ylabel(f"H{h}", rotation=0, labelpad=16, fontsize=12, color=ENCRE, va="center")
axes[0, 1].set_title("Normal", loc="left", fontsize=12)
axes[0, 5].set_title("Tumoral", loc="left", fontsize=12)
fig.suptitle("Les couleurs changent d'un hôpital à l'autre, pour un même type de tissu", x=0.07, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))
sauver(fig, "patchs_par_hopital")
plt.show()

## 3. Mesurer les écarts de couleur (tâche 6)

On tire 500 patchs par hôpital et par classe, puis on calcule leur couleur moyenne en RGB, HSV et Lab. La distance Delta-E se lit dans l'espace Lab : plus elle est grande, plus la différence est visible. Un écart d'environ 2 à 3 est à peine perceptible.

Limite : la moyenne d'un patch mêle le tissu et le fond clair. Elle décrit bien l'allure d'un hôpital, mais pas la coloration seule.

In [ ]:
ech = df.groupby(["center", "label"]).sample(500, random_state=SEED).reset_index(drop=True)
imgs = [image(b) for b in ech.image]

def moyennes(i):
    hsv = np.asarray(Image.fromarray(i).convert("HSV"), dtype=float).reshape(-1, 3).mean(0)
    return np.concatenate([i.reshape(-1, 3).mean(0), hsv, pp.rgb_vers_lab(i).reshape(-1, 3).mean(0)])

CANAUX = ["Rouge", "Vert", "Bleu", "Teinte", "Saturation", "Valeur", "L*", "a*", "b*"]
couleurs = pd.DataFrame([moyennes(i) for i in imgs], columns=CANAUX)
couleurs[["center", "label", "slide"]] = ech[["center", "label", "slide"]].values
resume = couleurs.groupby("center")[CANAUX].median().round(1)
resume.to_csv(f"{SORTIE}/couleur_mediane_par_hopital.csv")
display(resume)

fig, axes = plt.subplots(3, 3, figsize=(10, 6.5), sharey=True)
for ax, canal in zip(axes.ravel(), CANAUX):
    for k, h in enumerate(HOPITAUX):
        q1, med, q3 = np.percentile(couleurs.loc[couleurs.center == h, canal], [25, 50, 75])
        ax.plot([q1, q3], [k, k], color=BLEU, lw=2, solid_capstyle="round")
        ax.plot(med, k, "o", ms=8, color=BLEU, mec="#fcfcfb", mew=2)
    ax.set_title(canal, loc="left", fontsize=10.5)
    ax.set_yticks(range(5), [f"H{h}" for h in HOPITAUX]); ax.invert_yaxis() if ax is axes[0, 0] else None
    ax.grid(axis="x", color=GRILLE, lw=0.8); ax.set_axisbelow(True); ax.tick_params(length=0)
fig.suptitle("Couleur moyenne des patchs par hôpital : médiane et moitié centrale", x=0.02, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))
sauver(fig, "couleur_par_hopital")
plt.show()

In [ ]:
def distances(cols, centres):
    s = pd.DataFrame(cols, columns=["L", "a", "b"]).assign(center=list(centres))
    return pp.distances_lab(s)

def hors_diagonale(d):
    return d.values[np.triu_indices(len(d), 1)]

LAB = ["L*", "a*", "b*"]
d_avant = distances(couleurs[LAB].values, couleurs.center)
d_avant.round(1).to_csv(f"{SORTIE}/delta_e_entre_hopitaux.csv")
print("Delta-E entre hôpitaux"); display(d_avant.round(1))

connus = couleurs[couleurs.center != 2]
ecart_classe = {f"H{h}": np.linalg.norm(g[g.label == 1][LAB].mean() - g[g.label == 0][LAB].mean()) for h, g in connus.groupby("center")}
print(f"Écart moyen entre hôpitaux : {hors_diagonale(d_avant).mean():.1f}  (de {hors_diagonale(d_avant).min():.1f} à {hors_diagonale(d_avant).max():.1f})")
print("Écart entre normal et tumoral dans un même hôpital :", {k: round(v, 1) for k, v in ecart_classe.items()})

### La couleur moyenne trahit-elle l'hôpital ?

Test simple : avec seulement trois nombres (L, a, b moyens), une régression logistique devine-t-elle l'hôpital ? Devine-t-elle si le patch est tumoral ? Les lames testées ne servent jamais à l'entraînement de ce petit modèle, pour éviter toute fuite. H2 est exclu de la seconde question.

Ce n'est pas la référence de P2 : ce test sert seulement à comparer deux signaux.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
acc = cross_val_score(m, couleurs[LAB], couleurs.center.astype(int), groups=couleurs.slide, cv=GroupKFold(5))
auc = cross_val_score(m, connus[LAB], connus.label.astype(int), groups=connus.slide, cv=GroupKFold(5), scoring="roc_auc")
print(f"Deviner l'hôpital (5 choix, hasard 20 %) : accuracy {acc.mean():.1%} (écart-type {acc.std():.1%})")
print(f"Deviner tumoral ou non (hasard 0,50) : ROC-AUC {auc.mean():.2f} (écart-type {auc.std():.2f})")
pd.DataFrame({"hopital_accuracy": [acc.mean()], "tumoral_auc": [auc.mean()]}).round(3).to_csv(f"{SORTIE}/signal_couleur.csv", index=False)

## 4. Essayer `preprocessing.py` sur de vraies images (tâche 3)

Les références de couleur s'apprennent sur `train_interne` uniquement. On regarde le rendu, le taux d'échec et le temps de calcul. Réduire l'écart de couleur ne prouve pas que le modèle sera meilleur : cela se mesure à la tâche 9.

In [ ]:
ref_df = df[df.split_equipe == "train_interne"].groupby("center").sample(100, random_state=SEED)
ref_imgs, ref_centres = [image(b) for b in ref_df.image], ref_df.center.tolist()
reinhard = pp.Reinhard().fit(ref_imgs, ref_centres)
macenko = pp.Macenko().fit(ref_imgs, ref_centres)
print("Vecteurs de coloration appris (H, E) :"); print(macenko.stains.round(3))

fig, axes = plt.subplots(5, 6, figsize=(9, 7.8))
for r, h in enumerate(HOPITAUX):
    t = df[df.center == h].sample(2, random_state=SEED + 1)
    for p, octets in enumerate(t.image):
        i = image(octets)
        for c, v in enumerate([i, reinhard(i), macenko(i)]):
            ax = axes[r, p * 3 + c]
            ax.imshow(v); ax.set_xticks([]); ax.set_yticks([])
            for bord in ax.spines.values():
                bord.set_visible(False)
            if r == 0:
                ax.set_title(["Origine", "Reinhard", "Macenko"][c], fontsize=10)
    axes[r, 0].set_ylabel(f"H{h}", rotation=0, labelpad=16, fontsize=12, color=ENCRE, va="center")
fig.suptitle("Deux patchs par hôpital, avant et après normalisation", x=0.07, ha="left", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.96))
sauver(fig, "normalisation_avant_apres")
plt.show()

In [ ]:
sous = ech.groupby(["center", "label"]).sample(100, random_state=SEED)
sous_imgs, sous_centres = [image(b) for b in sous.image], sous.center.values

def appliquer(f):
    t0 = time.perf_counter(); sorties = [f(i) for i in sous_imgs]
    return sorties, (time.perf_counter() - t0) / len(sous_imgs) * 1000

def lab_moyen(liste):
    return np.stack([pp.rgb_vers_lab(i).reshape(-1, 3).mean(0) for i in liste])

macenko.n_echecs = 0
res_r, ms_r = appliquer(reinhard)
res_m, ms_m = appliquer(macenko)
bilan = []
for nom, liste, ms, ech_ in [("Origine", sous_imgs, 0.0, 0), ("Reinhard", res_r, ms_r, 0), ("Macenko", res_m, ms_m, macenko.n_echecs)]:
    d = hors_diagonale(distances(lab_moyen(liste), sous_centres))
    bilan.append({"technique": nom, "delta_e_moyen": d.mean(), "delta_e_max": d.max(), "ms_par_patch": ms, "patchs_en_echec": ech_})
bilan = pd.DataFrame(bilan).round(2)
bilan.to_csv(f"{SORTIE}/normalisation_bilan.csv", index=False)
display(bilan)
print(f"{len(sous_imgs)} patchs. Taux d'échec Macenko : {macenko.n_echecs / len(sous_imgs):.1%}")
print(f"Macenko sur 24 000 patchs, à chaque époque : environ {ms_m * 24000 / 60000:.0f} min")

In [ ]:
import torch
pil = Image.fromarray(imgs[0])
for mode in pp.MODES:
    ref = {"reinhard": reinhard, "macenko": macenko}.get(mode)
    entr, ev = pp.get_transforms(mode, ref)
    for t in (entr, ev):
        x = t(pil)
        assert x.shape == (3, 96, 96) and torch.isfinite(x).all(), mode
    print(f"{mode:12s} valide")
print("Fichiers enregistrés :", sorted(os.listdir(SORTIE)))